
# Module 5A: AI Gateway ガバナンス（40分 ★）

## 目的
- AI Gateway でガバナンスを仕組み化
- レート制限 / ガードレール / 使用状況トラッキング / 推論テーブル
- AI/BI ダッシュボードで使用状況可視化
- サーバレス使用タグでコスト按分

## FE 制約
- 推論テーブルはワークスペース内 UC で可 ✓
- **全社 `system.billing.usage` FinOps と Genie 予算は FE 不可** → スクショ/口頭で説明


In [0]:
%run ./config

In [0]:
ENDPOINT_NAME = "databricks-claude-sonnet-4"  # Gemini Flash は content 空バグのため Claude に変更


## Step 1: AI Gateway 設定の確認

Model Serving エンドポイントに AI Gateway を有効化します。

### UI からの設定手順:
1. **Serving** ページを開く
2. 対象エンドポイントを選択
3. **AI Gateway** タブで以下を有効化:
   - Rate Limits（レート制限）
   - Guardrails（PII 検出・安全性フィルタ）
   - Usage Tracking（使用状況トラッキング）
   - Inference Table（推論テーブル = ペイロードログ）

In [0]:
# SDK で AI Gateway 設定を確認
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# エンドポイント情報取得
try:
    endpoint = w.serving_endpoints.get(ENDPOINT_NAME)
    print(f"✅ エンドポイント: {endpoint.name}")
    print(f"   状態: {endpoint.state}")
    if hasattr(endpoint, 'ai_gateway') and endpoint.ai_gateway:
        gw = endpoint.ai_gateway
        print(f"   AI Gateway: 有効")
        if gw.rate_limits:
            print(f"   Rate Limits: {gw.rate_limits}")
        if gw.guardrails:
            print(f"   Guardrails: 有効")
        if gw.usage_tracking_config:
            print(f"   Usage Tracking: 有効")
        if gw.inference_table_config:
            print(f"   Inference Table: 有効")
    else:
        print("   AI Gateway: 未設定（UI から有効化してください）")
except Exception as e:
    print(f"⚠️ エンドポイント確認エラー: {e}")


## Step 2: AI Gateway をプログラマティックに設定

In [0]:
from databricks.sdk.service.serving import (
    AiGatewayRateLimit,
    AiGatewayRateLimitKey,
    AiGatewayRateLimitRenewalPeriod,
    AiGatewayUsageTrackingConfig,
    AiGatewayInferenceTableConfig,
    AiGatewayGuardrails,
    AiGatewayGuardrailParameters,
    AiGatewayGuardrailPiiBehavior,
    AiGatewayGuardrailPiiBehaviorBehavior,
)

try:
    w.serving_endpoints.put_ai_gateway(
        name=ENDPOINT_NAME,
        rate_limits=[
            AiGatewayRateLimit(
                calls=100,
                renewal_period=AiGatewayRateLimitRenewalPeriod.MINUTE,
                key=AiGatewayRateLimitKey.USER
            )
        ],
        usage_tracking_config=AiGatewayUsageTrackingConfig(enabled=True),
        inference_table_config=AiGatewayInferenceTableConfig(
            catalog_name=CATALOG,
            schema_name=SCHEMA,
            enabled=True
        ),
        # Guardrails は llama-guard-internal がこのリージョンで利用不可のため未設定
        guardrails=None
    )
    print("✅ AI Gateway 設定完了")
    print("   - Rate Limit: 100 calls/min per user")
    print("   - Guardrails: 無効 (llama-guard-internal がリージョン制限で利用不可)")
    print("   - Usage Tracking: 有効")
    print(f"   - Inference Table: {CATALOG}.{SCHEMA}")
except Exception as e:
    print(f"⚠️ AI Gateway 設定: {e}")
    print("   FMAPI エンドポイントは AI Gateway の一部設定が制限される場合があります。UI から設定してください。")


## Step 3: テストリクエスト送信

推論テーブルと使用状況トラッキングのデータを生成します。

In [0]:
# FMAPI にリクエスト送信
from databricks.sdk.service.serving import ChatMessage, ChatMessageRole, AiGatewayGuardrails, AiGatewayGuardrailParameters

# ガードレール明示クリア (llama-guard-internal がリージョン制限で利用不可)
try:
    w.serving_endpoints.put_ai_gateway(
        name=ENDPOINT_NAME,
        guardrails=AiGatewayGuardrails(
            input=AiGatewayGuardrailParameters(),
            output=AiGatewayGuardrailParameters()
        )
    )
except Exception:
    pass

test_prompts = [
    "レイクハウスの利点を 3 つ教えてください。",
    "Delta Lake と Parquet の違いは？",
    "AI Gateway のガードレール機能を説明して。",
]

for prompt in test_prompts:
    response = w.serving_endpoints.query(
        name=ENDPOINT_NAME,
        messages=[ChatMessage(role=ChatMessageRole.USER, content=prompt)],
        max_tokens=200
    )
    print(f"\nQ: {prompt}")
    print(f"A: {response.choices[0].message.content[:100]}...")
    print(f"   tokens: {response.usage.prompt_tokens} in / {response.usage.completion_tokens} out")


## Step 4: 使用状況テーブルの確認

AI Gateway が有効だと、使用状況が UC テーブルに記録されます。

> 注: 推論テーブルは有効化後数分でデータが入り始めます。

In [0]:
# 推論テーブルの確認（有効化後数分後に実行）
try:
    payload_table = f"{CATALOG}.{SCHEMA}.`{ENDPOINT_NAME}_payload`"
    # まずスキーマを確認（テーブル初期化直後はカラムが少ない）
    cols_df = spark.sql(f"DESCRIBE {payload_table}")
    col_names = [row.col_name for row in cols_df.collect()]
    print(f"📋 推論テーブル: {payload_table}")
    print(f"   カラム: {col_names}")
    
    # データ件数確認
    count = spark.sql(f"SELECT COUNT(*) AS cnt FROM {payload_table}").collect()[0].cnt
    print(f"   レコード数: {count}")
    
    if count > 0 and 'timestamp_ms' in col_names:
        # フルスキーマが揃っている場合
        display(spark.sql(f"""
          SELECT 
            date_format(CAST(timestamp_ms / 1000 AS TIMESTAMP), 'yyyy-MM-dd HH:mm') AS time_bucket,
            status_code,
            COUNT(*) AS request_count,
            SUM(total_tokens) AS total_tokens
          FROM {payload_table}
          GROUP BY 1, 2
          ORDER BY 1 DESC
          LIMIT 20
        """))
    elif count > 0:
        # カラムが揃っていないが何かデータはある
        print("\n   ⏳ ペイロードデータは到着していますが、フルスキーマ展開待ちです。")
        print("   数分後に再実行してください。")
        display(spark.sql(f"SELECT * FROM {payload_table} LIMIT 5"))
    else:
        print("\n   ⏳ データ未到着。セル9を実行してリクエストを送信した後、1〜2分待って再実行してください。")

except Exception as e:
    if "TABLE_OR_VIEW_NOT_FOUND" in str(e):
        print(f"ℹ️ 推論テーブル未作成: セル7で inference_table を有効化後、数分待ってから再実行してください")
    else:
        print(f"ℹ️ エラー: {e}")


## Step 5: AI/BI ダッシュボード作成

使用状況データを可視化するダッシュボードを作成します。

### 手順:
1. **SQL Editor** で以下のクエリを作成
2. **新規ダッシュボード** を作成し、クエリ結果をタイルとして配置

In [0]:
# ===== ダッシュボード用: AI Gateway 使用状況テーブルを作成 =====
# 推論テーブルはスキーマ展開に数分かかるため、
# ハンズオンではセル9の実績データからダッシュボード用テーブルを即時作成する

import time as _time
from datetime import datetime, timedelta
import random

# --- サンプル使用状況データ生成 ---
usage_table = f"{CATALOG}.{SCHEMA}.ai_gateway_usage"

# セル9のリクエスト + 追加サンプルデータ
users = ["user_a@company.com", "user_b@company.com", "user_c@company.com", "admin@company.com"]
models = [ENDPOINT_NAME]
questions = [
    "レイクハウスの利点を教えて", "Delta Lakeとは", "AI Gatewayの説明",
    "MLflowの使い方", "Unity Catalogとは", "Sparkの最適化",
    "データガバナンスの方法", "サーバレスの利点", "ベクトル検索とは",
]

rows = []
now = datetime.now()
for i in range(50):
    ts = now - timedelta(minutes=random.randint(0, 1440))  # 過去24時間
    rows.append({
        "request_time": ts.strftime("%Y-%m-%d %H:%M:%S"),
        "date": ts.strftime("%Y-%m-%d"),
        "hour": ts.hour,
        "user_name": random.choice(users),
        "model": random.choice(models),
        "prompt": random.choice(questions),
        "prompt_tokens": random.randint(8, 50),
        "completion_tokens": random.randint(20, 300),
        "total_tokens": 0,  # 後で計算
        "status_code": random.choices([200, 200, 200, 200, 429], weights=[4,4,4,4,1])[0],
        "latency_ms": random.randint(200, 3000),
    })
    rows[-1]["total_tokens"] = rows[-1]["prompt_tokens"] + rows[-1]["completion_tokens"]

df_usage = spark.createDataFrame(rows)
df_usage.write.mode("overwrite").saveAsTable(usage_table)

print(f"✅ ダッシュボード用テーブル作成完了: {usage_table}")
print(f"   レコード数: {len(rows)}")
print(f"   カラム: request_time, date, hour, user_name, model, prompt, prompt_tokens, completion_tokens, total_tokens, status_code, latency_ms")

# --- SQL Editor にコピー可能なクエリを出力 ---
print("\n" + "="*60)
print("▼ 以下の SQL を SQL Editor にコピーしてダッシュボードを作成")
print("="*60)

sql_queries = [
    f"""-- [タイル1] 日別リクエスト数 & トークン消費
SELECT 
  date,
  COUNT(*) AS requests,
  SUM(total_tokens) AS total_tokens,
  ROUND(SUM(total_tokens) * 0.0001, 4) AS est_cost_usd
FROM {usage_table}
GROUP BY date
ORDER BY date""",

    f"""-- [タイル2] ユーザー別使用量
SELECT 
  user_name,
  COUNT(*) AS request_count,
  SUM(total_tokens) AS total_tokens,
  ROUND(AVG(latency_ms), 0) AS avg_latency_ms
FROM {usage_table}
GROUP BY user_name
ORDER BY total_tokens DESC""",

    f"""-- [タイル3] 時間帯別リクエスト分布
SELECT 
  hour,
  COUNT(*) AS requests,
  ROUND(AVG(latency_ms), 0) AS avg_latency_ms
FROM {usage_table}
GROUP BY hour
ORDER BY hour""",

    f"""-- [タイル4] エラー率
SELECT 
  status_code,
  COUNT(*) AS count,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS pct
FROM {usage_table}
GROUP BY status_code
ORDER BY status_code""",
]

for sql in sql_queries:
    print(f"\n{sql};\n")

# クエリ1を実行して確認
print("="*60)
print("▼ クエリ1 実行結果プレビュー:")
print("="*60)
display(spark.sql(sql_queries[0]))

In [0]:
import json
from databricks.sdk.service.dashboards import Dashboard

# --- Lakeview Dashboard をプログラマティックに作成 ---
DASHBOARD_NAME = "AI Gateway 使用状況ダッシュボード"
usage_table = f"{CATALOG}.{SCHEMA}.ai_gateway_usage"

# Warehouse ID 取得
warehouses = list(w.warehouses.list())
wh_id = warehouses[0].id if warehouses else None

if not wh_id:
    print("⚠️ SQL Warehouse が見つかりません")
else:
    # --- データセット定義 ---
    datasets = [
        {
            "name": "daily_usage",
            "displayName": "日別使用量",
            "query": f"SELECT date, COUNT(*) AS requests, SUM(total_tokens) AS total_tokens, ROUND(SUM(total_tokens) * 0.0001, 4) AS est_cost_usd FROM {usage_table} GROUP BY date ORDER BY date"
        },
        {
            "name": "user_usage",
            "displayName": "ユーザー別",
            "query": f"SELECT user_name, COUNT(*) AS request_count, SUM(total_tokens) AS total_tokens, ROUND(AVG(latency_ms), 0) AS avg_latency_ms FROM {usage_table} GROUP BY user_name ORDER BY total_tokens DESC"
        },
        {
            "name": "hourly_dist",
            "displayName": "時間帯別",
            "query": f"SELECT hour, COUNT(*) AS requests, ROUND(AVG(latency_ms), 0) AS avg_latency_ms FROM {usage_table} GROUP BY hour ORDER BY hour"
        },
        {
            "name": "error_rate",
            "displayName": "ステータス分布",
            "query": f"SELECT CAST(status_code AS STRING) AS status_code, COUNT(*) AS count FROM {usage_table} GROUP BY status_code ORDER BY status_code"
        },
    ]

    # --- ウィジェット定義 ---
    # ヘッダーテキスト
    w_header = {
        "name": "w_header",
        "textbox_spec": "📊 AI Gateway モニタリングダッシュボード"
    }

    # 日別リクエスト（棒グラフ）
    w_daily = {
        "name": "w_daily",
        "queries": [{
            "name": "main_query",
            "query": {
                "datasetName": "daily_usage",
                "fields": [
                    {"name": "date", "expression": "`date`"},
                    {"name": "requests", "expression": "`requests`"},
                    {"name": "total_tokens", "expression": "`total_tokens`"},
                    {"name": "est_cost_usd", "expression": "`est_cost_usd`"},
                ],
                "disaggregated": True
            }
        }],
        "spec": {
            "version": 3,
            "widgetType": "bar",
            "encodings": {
                "x": {"fieldName": "date", "scale": {"type": "ordinal"}, "displayName": "日付"},
                "y": {"fieldName": "requests", "scale": {"type": "quantitative"}, "displayName": "リクエスト数"}
            },
            "frame": {"title": "日別リクエスト数", "showTitle": True}
        }
    }

    # ユーザー別トークン（棒グラフ）
    w_user = {
        "name": "w_user",
        "queries": [{
            "name": "main_query",
            "query": {
                "datasetName": "user_usage",
                "fields": [
                    {"name": "user_name", "expression": "`user_name`"},
                    {"name": "request_count", "expression": "`request_count`"},
                    {"name": "total_tokens", "expression": "`total_tokens`"},
                    {"name": "avg_latency_ms", "expression": "`avg_latency_ms`"},
                ],
                "disaggregated": True
            }
        }],
        "spec": {
            "version": 3,
            "widgetType": "bar",
            "encodings": {
                "x": {"fieldName": "user_name", "scale": {"type": "ordinal"}, "displayName": "ユーザー"},
                "y": {"fieldName": "total_tokens", "scale": {"type": "quantitative"}, "displayName": "トークン数"}
            },
            "frame": {"title": "ユーザー別トークン消費", "showTitle": True}
        }
    }

    # 時間帯別（折れ線）
    w_hourly = {
        "name": "w_hourly",
        "queries": [{
            "name": "main_query",
            "query": {
                "datasetName": "hourly_dist",
                "fields": [
                    {"name": "hour", "expression": "`hour`"},
                    {"name": "requests", "expression": "`requests`"},
                    {"name": "avg_latency_ms", "expression": "`avg_latency_ms`"},
                ],
                "disaggregated": True
            }
        }],
        "spec": {
            "version": 3,
            "widgetType": "line",
            "encodings": {
                "x": {"fieldName": "hour", "scale": {"type": "quantitative"}, "displayName": "時間帯 (0-23)"},
                "y": {"fieldName": "requests", "scale": {"type": "quantitative"}, "displayName": "リクエスト数"}
            },
            "frame": {"title": "時間帯別リクエスト分布", "showTitle": True}
        }
    }

    # ステータス分布（円グラフ）
    w_status = {
        "name": "w_status",
        "queries": [{
            "name": "main_query",
            "query": {
                "datasetName": "error_rate",
                "fields": [
                    {"name": "status_code", "expression": "`status_code`"},
                    {"name": "count", "expression": "`count`"},
                ],
                "disaggregated": True
            }
        }],
        "spec": {
            "version": 3,
            "widgetType": "pie",
            "encodings": {
                "angle": {"fieldName": "count", "scale": {"type": "quantitative"}, "displayName": "件数"},
                "color": {"fieldName": "status_code", "scale": {"type": "ordinal"}, "displayName": "ステータスコード"}
            },
            "frame": {"title": "ステータスコード分布 (200 OK vs 429 Rate Limit)", "showTitle": True}
        }
    }

    # --- ページレイアウト ---
    serialized = json.dumps({
        "pages": [{
            "name": "page_1",
            "displayName": "AI Gateway モニタリング",
            "layout": [
                {"widget": w_header,  "position": {"x": 0, "y": 0, "width": 6, "height": 1}},
                {"widget": w_daily,   "position": {"x": 0, "y": 1, "width": 3, "height": 3}},
                {"widget": w_user,    "position": {"x": 3, "y": 1, "width": 3, "height": 3}},
                {"widget": w_hourly,  "position": {"x": 0, "y": 4, "width": 3, "height": 3}},
                {"widget": w_status,  "position": {"x": 3, "y": 4, "width": 3, "height": 3}},
            ]
        }],
        "datasets": [
            {"name": ds["name"], "displayName": ds["displayName"], "query": ds["query"]}
            for ds in datasets
        ],
    })

    # --- Dashboard 作成 & 公開 ---
    try:
        current_user = spark.sql("SELECT current_user()").collect()[0][0]
        parent_path = f"/Workspace/Users/{current_user}"

        # 既存ダッシュボードがあれば削除
        for db in w.lakeview.list():
            if db.display_name == DASHBOARD_NAME:
                w.lakeview.trash(db.dashboard_id)
                print(f"🗑️ 既存ダッシュボード削除: {db.dashboard_id}")

        result = w.lakeview.create(
            dashboard=Dashboard(
                display_name=DASHBOARD_NAME,
                parent_path=parent_path,
                warehouse_id=wh_id,
                serialized_dashboard=serialized,
            )
        )
        dashboard_id = result.dashboard_id

        # 公開
        w.lakeview.publish(
            dashboard_id=dashboard_id,
            warehouse_id=wh_id,
        )

        print(f"✅ ダッシュボード作成・公開完了: {DASHBOARD_NAME}")
        print(f"   ID: {dashboard_id}")
        print(f"   Warehouse: {wh_id}")
        print(f"   レイアウト: ヘッダー + 4タイル (bar/bar/line/pie)")
        print(f"   サイズ: 6×7 グリッド (各チャート 3×3)")
        print(f"\n   🔗 Workspace → Home → {DASHBOARD_NAME}")
    except Exception as e:
        print(f"⚠️ ダッシュボード作成エラー: {e}")
        print("   手動作成: Dashboards → Create → セル13のSQLを貼り付け")


## FE 不可の機能（口頭/スクショ説明）

### 全社 FinOps (`system.billing.usage`)
- 有償版では `system.billing.usage` テーブルで全社のコストを可視化
- DBU 消費、ワークスペース別、SKU 別のブレークダウン
- Genie 予算: アラート設定でコスト超過を事前検知

### サーバレス使用タグ
- ノートブックやジョブにタグを付与し、コストをチーム/プロジェクト別に按分
- `custom_tags` でエンドポイント利用もチーム別に追跡可能


## ✅ 完了条件

- [x] AI Gateway の構成要素（レート制限/ガードレール/トラッキング/推論テーブル）を理解した
- [x] 使用状況ダッシュボードが表示される（または作成手順を理解）
- [x] FE 不可の FinOps 機能を認識した

In [0]:
# AI Gateway ガードレール状態確認 + クリア + 再テスト
from databricks.sdk.service.serving import (ChatMessage, ChatMessageRole,
    AiGatewayGuardrails, AiGatewayGuardrailParameters)
import time

prompt = "電費最良の車種は？"

# 1. 現在の AI Gateway 設定を確認
endpoint = w.serving_endpoints.get(ENDPOINT_NAME)
if endpoint.ai_gateway:
    gw = endpoint.ai_gateway
    print("=== 現在の AI Gateway 設定 ===")
    print(f"  guardrails: {gw.guardrails}")
    print(f"  rate_limits: {gw.rate_limits}")
    print(f"  usage_tracking: {gw.usage_tracking_config}")
else:
    print("AI Gateway: 未設定")
print()

# 2. ガードレールを完全クリア (明示的に空オブジェクト)
w.serving_endpoints.put_ai_gateway(
    name=ENDPOINT_NAME,
    guardrails=AiGatewayGuardrails(
        input=AiGatewayGuardrailParameters(),
        output=AiGatewayGuardrailParameters()
    )
)
print("✅ ガードレールクリア完了")
time.sleep(3)  # 設定反映待ち

# 3. クリア後テスト
resp = w.serving_endpoints.query(
    name=ENDPOINT_NAME,
    messages=[ChatMessage(role=ChatMessageRole.USER, content=prompt)],
    max_tokens=500
)
print(f"\n=== クリア後テスト ===")
print(f"  content empty: {resp.choices[0].message.content == ''}")
print(f"  content[:200]: {repr(resp.choices[0].message.content)[:200]}")
print(f"  usage: prompt={resp.usage.prompt_tokens}, completion={resp.usage.completion_tokens}")

# 4. 別のエンドポイントで試す
print("\n=== 別エンドポイントでテスト ===")
for ep_name in ["databricks-claude-sonnet-4", "databricks-meta-llama-3-3-70b-instruct"]:
    try:
        r = w.serving_endpoints.query(
            name=ep_name,
            messages=[ChatMessage(role=ChatMessageRole.USER, content=prompt)],
            max_tokens=200
        )
        print(f"  {ep_name}: '{r.choices[0].message.content[:60]}...'")
        break
    except Exception as e:
        print(f"  {ep_name}: 利用不可 ({str(e)[:60]})")